# Phase 3 — the look-alike problem

Loads the **trained** Phase 2 checkpoint (no retraining, ~10 min) and:

1. splits top-1 errors into *look-alike* (right category, wrong product) and
   *off-target*, with examples;
2. tries test-time augmentation and alpha-weighted query expansion;
3. reports before/after Recall@1 **on the look-alike cases specifically**.

**Setup:** attach the SOP dataset as usual, *and* attach the Phase 2 run's
output so the checkpoint is reachable — *Add Input → Your Work →* the notebook
version that trained `triplet_hard`. Accelerator: GPU.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import json, pathlib, sys

# The vpse package, embedded so this notebook is self-contained.
_FILES = json.loads(r'''
{
"vpse/__init__.py": "",
"vpse/analysis/__init__.py": "",
"vpse/analysis/errors.py": "\"\"\"Phase 3 error analysis: where does top-1 go wrong, and how?\n\nSOP labels each image with a product (class_id) and a category\n(super_class_id). Splitting top-1 errors by whether the retrieved item shares\nthe query's category separates the two failure modes:\n\n  look-alike  - right kind of thing, wrong product (a different black saddle)\n  off-target  - wrong kind of thing entirely\n\nThe look-alike share is the number Phase 3 is trying to move.\n\"\"\"\nimport numpy as np\n\n\ndef top1_errors(nbrs: np.ndarray, labels: np.ndarray) -> np.ndarray:\n    \"\"\"Boolean mask of queries whose nearest neighbour is the wrong product.\"\"\"\n    return labels[nbrs[:, 0]] != labels\n\n\ndef error_breakdown(nbrs: np.ndarray, labels: np.ndarray,\n                    super_labels: np.ndarray) -> dict:\n    wrong = top1_errors(nbrs, labels)\n    same_cat = super_labels[nbrs[:, 0]] == super_labels\n    lookalike = wrong & same_cat\n    n_err = int(wrong.sum())\n    return {\n        \"n_queries\": int(len(labels)),\n        \"R@1\": float(1.0 - wrong.mean()),\n        \"n_top1_errors\": n_err,\n        \"lookalike_errors\": int(lookalike.sum()),\n        \"offtarget_errors\": int((wrong & ~same_cat).sum()),\n        \"lookalike_share_of_errors\": float(lookalike.sum() / n_err) if n_err else 0.0,\n    }\n\n\ndef hard_queries(nbrs: np.ndarray, labels: np.ndarray,\n                 super_labels: np.ndarray) -> np.ndarray:\n    \"\"\"Indices of look-alike failures: top-1 is the wrong product, same category.\"\"\"\n    wrong = top1_errors(nbrs, labels)\n    same_cat = super_labels[nbrs[:, 0]] == super_labels\n    return np.flatnonzero(wrong & same_cat)\n\n\ndef recall_on_subset(nbrs: np.ndarray, labels: np.ndarray, subset: np.ndarray,\n                     ks=(1, 5, 10)) -> dict:\n    \"\"\"Recall@k restricted to the given query indices.\"\"\"\n    if len(subset) == 0:\n        return {f\"R@{k}\": float(\"nan\") for k in ks}\n    hits = labels[nbrs[subset]] == labels[subset][:, None]\n    return {f\"R@{k}\": float(hits[:, :k].any(axis=1).mean()) for k in ks}\n",
"vpse/analysis/grids.py": "\"\"\"Query -> top-k result grids.\"\"\"\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom PIL import Image\n\n\ndef retrieval_grid(ds, nbrs: np.ndarray, labels: np.ndarray, queries,\n                   out_path: Path, k: int = 5, title: str | None = None):\n    queries = list(queries)\n    fig, axes = plt.subplots(len(queries), k + 1,\n                             figsize=(2.2 * (k + 1), 2.4 * len(queries)),\n                             squeeze=False)\n    for r, q in enumerate(queries):\n        for c, idx in enumerate([q] + nbrs[q, :k].tolist()):\n            ax = axes[r][c]\n            img = Image.open(ds.data_root / ds.df.iloc[idx][\"path\"]).convert(\"RGB\")\n            ax.imshow(img)\n            ax.axis(\"off\")\n            if c == 0:\n                ax.set_title(\"query\", fontsize=9)\n            else:\n                hit = labels[idx] == labels[q]\n                ax.set_title(\"match\" if hit else \"wrong\", fontsize=9,\n                             color=\"green\" if hit else \"red\")\n    if title:\n        fig.suptitle(title, fontsize=12)\n    fig.tight_layout()\n    out_path = Path(out_path)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(out_path, dpi=120)\n    plt.close(fig)\n    return out_path\n",
"vpse/config.py": "from dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n@dataclass\nclass Config:\n    # paths\n    data_root: Path = Path(\"data/Stanford_Online_Products\")\n    cache_dir: Path = Path(\"cache\")        # cached embeddings\n    results_dir: Path = Path(\"results\")\n\n    # model\n    embedding_dim: int = 512\n    freeze_backbone: bool = False          # True for the Phase 1 baseline\n\n    # data\n    image_size: int = 224\n    num_workers: int = 2\n\n    # training (Phase 2)\n    loss: str = \"triplet_hard\"             # triplet_random | triplet_hard | arcface\n    epochs: int = 30\n    lr_head: float = 1e-3\n    lr_backbone: float = 1e-5\n    weight_decay: float = 1e-4\n    triplet_margin: float = 0.2\n    arcface_scale: float = 30.0\n    arcface_margin: float = 0.3\n    # PK batches: P classes x K images. Bigger batch => better in-batch mining.\n    batch_p: int = 16\n    batch_k: int = 4\n\n    # retrieval / eval\n    recall_ks: tuple = (1, 5, 10)\n    # a full eval re-embeds the whole test split; don't do it every epoch\n    eval_every: int = 3\n\n    device: str = \"cuda\"\n\n    def __post_init__(self):\n        self.data_root = Path(self.data_root)\n        self.cache_dir = Path(self.cache_dir)\n        self.results_dir = Path(self.results_dir)\n",
"vpse/data/__init__.py": "",
"vpse/data/samplers.py": "\"\"\"PK batch sampler: each batch holds P classes x K images per class.\n\nIn-batch mining (batch-hard triplet) only works if every anchor has positives\nin the batch \u2014 random shuffling over 11k classes almost never provides them.\n\"\"\"\nimport random\nfrom collections import defaultdict\n\nimport numpy as np\nfrom torch.utils.data import Sampler\n\n\nclass PKSampler(Sampler):\n    def __init__(self, labels: np.ndarray, p: int, k: int):\n        self.p, self.k = p, k\n        self.index_by_label = defaultdict(list)\n        for idx, lab in enumerate(labels):\n            self.index_by_label[int(lab)].append(idx)\n        # classes with fewer than 2 images can't form positive pairs\n        self.usable = [l for l, idxs in self.index_by_label.items() if len(idxs) >= 2]\n        self.batches_per_epoch = len(labels) // (p * k)\n\n    def __len__(self):\n        return self.batches_per_epoch\n\n    def __iter__(self):\n        for _ in range(self.batches_per_epoch):\n            batch = []\n            for lab in random.sample(self.usable, self.p):\n                idxs = self.index_by_label[lab]\n                replace = len(idxs) < self.k\n                batch.extend(np.random.choice(idxs, self.k, replace=replace).tolist())\n            yield batch\n",
"vpse/data/sop.py": "\"\"\"Stanford Online Products dataset.\n\nThe official split files (Ebay_train.txt / Ebay_test.txt) list:\n    image_id class_id super_class_id path\nTrain uses class ids 1..11318, test uses 11319..22634 \u2014 products in the test\nset are never seen in training, which is what makes this retrieval rather\nthan classification.\n\"\"\"\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\ndef load_split(data_root: Path, split: str) -> pd.DataFrame:\n    \"\"\"split: 'train' or 'test'. Returns df with class_id remapped to 0..C-1.\"\"\"\n    f = Path(data_root) / f\"Ebay_{split}.txt\"\n    df = pd.read_csv(f, sep=\" \")\n    df.columns = [c.strip() for c in df.columns]\n    # contiguous labels for loss heads (e.g. ArcFace needs 0..C-1)\n    df[\"label\"] = df[\"class_id\"].astype(\"category\").cat.codes\n    return df\n\n\ndef train_transform(image_size: int = 224):\n    return transforms.Compose([\n        transforms.RandomResizedCrop(image_size, scale=(0.65, 1.0)),\n        transforms.RandomHorizontalFlip(),\n        transforms.ToTensor(),\n        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n    ])\n\n\ndef eval_transform(image_size: int = 224):\n    return transforms.Compose([\n        transforms.Resize(int(image_size * 256 / 224)),\n        transforms.CenterCrop(image_size),\n        transforms.ToTensor(),\n        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n    ])\n\n\nclass SOPDataset(Dataset):\n    def __init__(self, data_root: Path, split: str, transform):\n        self.data_root = Path(data_root)\n        self.df = load_split(data_root, split)\n        self.transform = transform\n        self.labels = self.df[\"label\"].to_numpy()\n        # category (12 of them) -- used by Phase 3 to separate look-alike\n        # errors (right category, wrong product) from off-target ones\n        self.super_labels = self.df[\"super_class_id\"].to_numpy()\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, i):\n        row = self.df.iloc[i]\n        img = Image.open(self.data_root / row[\"path\"]).convert(\"RGB\")\n        return self.transform(img), int(row[\"label\"])\n",
"vpse/losses/__init__.py": "",
"vpse/losses/arcface.py": "\"\"\"ArcFace (Deng et al. 2019): additive angular margin on a cosine classifier.\n\nUsed only at training time \u2014 at inference the class weights are discarded and\nretrieval runs on the embeddings.\n\"\"\"\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass ArcFaceHead(nn.Module):\n    def __init__(self, embedding_dim: int, n_classes: int,\n                 scale: float = 30.0, margin: float = 0.3):\n        super().__init__()\n        self.weight = nn.Parameter(torch.empty(n_classes, embedding_dim))\n        nn.init.xavier_uniform_(self.weight)\n        self.scale, self.margin = scale, margin\n        self.cos_m, self.sin_m = math.cos(margin), math.sin(margin)\n\n    def forward(self, emb: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:\n        # emb is already L2-normalized by the embedder\n        cos = F.linear(emb, F.normalize(self.weight, dim=1)).clamp(-1 + 1e-7, 1 - 1e-7)\n        sin = torch.sqrt(1.0 - cos ** 2)\n        cos_with_margin = cos * self.cos_m - sin * self.sin_m  # cos(theta + m)\n        # margin only makes sense while theta + m < pi; fall back otherwise\n        cos_target = torch.where(cos > math.cos(math.pi - self.margin),\n                                 cos_with_margin,\n                                 cos - self.margin * self.sin_m)\n        onehot = F.one_hot(labels, num_classes=self.weight.shape[0]).bool()\n        logits = torch.where(onehot, cos_target, cos) * self.scale\n        return F.cross_entropy(logits, labels)\n",
"vpse/losses/triplet.py": "\"\"\"Triplet losses over L2-normalized embeddings.\n\nBoth operate on a PK batch (see data/samplers.py). Distances are Euclidean;\nsince embeddings are unit-norm, d^2 = 2 - 2*cosine, so ranking is equivalent.\n\"\"\"\nimport torch\nimport torch.nn.functional as F\n\n\ndef _pairwise_dist(emb: torch.Tensor) -> torch.Tensor:\n    return torch.cdist(emb, emb, p=2)\n\n\ndef triplet_random(emb: torch.Tensor, labels: torch.Tensor, margin: float = 0.2):\n    \"\"\"Anchor-positive pairs from the batch, one *random* negative each.\"\"\"\n    dist = _pairwise_dist(emb)\n    same = labels[:, None] == labels[None, :]\n    eye = torch.eye(len(labels), dtype=torch.bool, device=emb.device)\n    pos_mask = same & ~eye\n    neg_mask = ~same\n\n    losses = []\n    for a in range(len(labels)):\n        pos_idx = pos_mask[a].nonzero(as_tuple=True)[0]\n        neg_idx = neg_mask[a].nonzero(as_tuple=True)[0]\n        if len(pos_idx) == 0 or len(neg_idx) == 0:\n            continue\n        p = pos_idx[torch.randint(len(pos_idx), (1,), device=emb.device)]\n        n = neg_idx[torch.randint(len(neg_idx), (1,), device=emb.device)]\n        losses.append(F.relu(dist[a, p] - dist[a, n] + margin))\n    return torch.cat(losses).mean() if losses else emb.sum() * 0.0\n\n\ndef triplet_batch_hard(emb: torch.Tensor, labels: torch.Tensor, margin: float = 0.2):\n    \"\"\"Batch-hard mining (Hermans et al. 2017): for each anchor take the\n    hardest (farthest) positive and hardest (closest) negative in the batch.\"\"\"\n    dist = _pairwise_dist(emb)\n    same = labels[:, None] == labels[None, :]\n    eye = torch.eye(len(labels), dtype=torch.bool, device=emb.device)\n\n    pos_dist = dist.masked_fill(~(same & ~eye), float(\"-inf\")).max(dim=1).values\n    neg_dist = dist.masked_fill(same, float(\"inf\")).min(dim=1).values\n\n    valid = torch.isfinite(pos_dist) & torch.isfinite(neg_dist)\n    return F.relu(pos_dist[valid] - neg_dist[valid] + margin).mean()\n",
"vpse/models/__init__.py": "",
"vpse/models/embedder.py": "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torchvision.models import resnet50, ResNet50_Weights\n\n\nclass Embedder(nn.Module):\n    \"\"\"ResNet50 backbone -> GAP -> linear head -> L2-normalized embedding.\n\n    With freeze_backbone=True and use_head=False this is the Phase 1 baseline:\n    raw pretrained pool5 features, no training at all.\n    \"\"\"\n\n    def __init__(self, embedding_dim: int = 512, freeze_backbone: bool = False,\n                 use_head: bool = True):\n        super().__init__()\n        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)\n        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # drop fc\n        self.use_head = use_head\n        self.head = nn.Linear(2048, embedding_dim) if use_head else nn.Identity()\n        if freeze_backbone:\n            for p in self.backbone.parameters():\n                p.requires_grad = False\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        feats = self.backbone(x).flatten(1)\n        emb = self.head(feats)\n        return F.normalize(emb, dim=1)\n",
"vpse/ood/__init__.py": "",
"vpse/ood/gate.py": "\"\"\"OOD refusal gate (Phase 4).\n\nScore a query by its similarity to the nearest gallery item; refuse when it\nfalls below a threshold. The threshold is chosen from the operating curve over\nin-catalog vs. out-of-catalog (e.g. random ImageNet) query sets.\n\"\"\"\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\n@dataclass\nclass OODGate:\n    threshold: float\n\n    def accept(self, top1_similarity: np.ndarray) -> np.ndarray:\n        \"\"\"True where the query should be answered, False where refused.\"\"\"\n        return top1_similarity >= self.threshold\n\n\ndef refusal_curve(in_catalog_sims: np.ndarray, ood_sims: np.ndarray,\n                  n_points: int = 200) -> dict:\n    \"\"\"Sweep thresholds over both distributions.\n\n    Returns arrays: threshold, ood_refused (TPR of refusal, higher=better),\n    valid_refused (false-refusal rate, lower=better).\n    \"\"\"\n    lo = min(in_catalog_sims.min(), ood_sims.min())\n    hi = max(in_catalog_sims.max(), ood_sims.max())\n    thresholds = np.linspace(lo, hi, n_points)\n    ood_refused = np.array([(ood_sims < t).mean() for t in thresholds])\n    valid_refused = np.array([(in_catalog_sims < t).mean() for t in thresholds])\n    return {\"threshold\": thresholds, \"ood_refused\": ood_refused,\n            \"valid_refused\": valid_refused}\n\n\ndef pick_threshold(curve: dict, max_valid_refused: float = 0.05) -> float:\n    \"\"\"Highest threshold that keeps false refusals under the budget.\"\"\"\n    ok = curve[\"valid_refused\"] <= max_valid_refused\n    return float(curve[\"threshold\"][ok][-1]) if ok.any() else float(curve[\"threshold\"][0])\n",
"vpse/retrieval/__init__.py": "",
"vpse/retrieval/eval.py": "\"\"\"Retrieval metrics for SOP-style evaluation.\n\nStandard SOP protocol: every test image is a query against all *other* test\nimages (leave-one-out gallery). A hit at k means any of the top-k neighbors\nshares the query's product class.\n\nThe neighbor search is brute-force. FAISS handles it on CPU; when a GPU is\navailable a chunked matmul is roughly two orders of magnitude faster, which\nmatters because training evaluates repeatedly.\n\"\"\"\nimport faiss\nimport numpy as np\nimport torch\n\n\ndef _neighbors_faiss(queries: np.ndarray, gallery: np.ndarray, k_max: int,\n                     exclude_self: bool) -> np.ndarray:\n    index = faiss.IndexFlatIP(gallery.shape[1])\n    index.add(gallery.astype(\"float32\"))\n    fetch = k_max + 1 if exclude_self else k_max\n    _, idx = index.search(queries.astype(\"float32\"), fetch)\n    if not exclude_self:\n        return idx.astype(np.int64)\n    out = np.empty((len(idx), k_max), dtype=np.int64)\n    for i, row in enumerate(idx):\n        out[i] = row[row != i][:k_max]\n    return out\n\n\ndef _neighbors_torch(queries: np.ndarray, gallery: np.ndarray, k_max: int,\n                     device: str, exclude_self: bool, chunk: int = 2048) -> np.ndarray:\n    q = torch.from_numpy(queries.astype(\"float32\")).to(device)\n    g = torch.from_numpy(gallery.astype(\"float32\")).to(device)\n    out = torch.empty((len(q), k_max), dtype=torch.int64, device=device)\n    for start in range(0, len(q), chunk):\n        stop = min(start + chunk, len(q))\n        sims = q[start:stop] @ g.T\n        if exclude_self:\n            # query i corresponds to gallery row i\n            rows = torch.arange(stop - start, device=device)\n            sims[rows, torch.arange(start, stop, device=device)] = float(\"-inf\")\n        out[start:stop] = sims.topk(k_max, dim=1).indices\n    return out.cpu().numpy()\n\n\ndef neighbors(queries: np.ndarray, gallery: np.ndarray, k_max: int,\n              exclude_self: bool = False, device: str | None = None) -> np.ndarray:\n    \"\"\"Top-k_max gallery indices per query. [Nq, k_max]\n\n    exclude_self assumes query i is gallery row i (leave-one-out).\n    \"\"\"\n    if device is None:\n        device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    if device == \"cpu\":\n        return _neighbors_faiss(queries, gallery, k_max, exclude_self)\n    return _neighbors_torch(queries, gallery, k_max, device, exclude_self)\n\n\ndef leave_one_out_neighbors(embeddings: np.ndarray, k_max: int,\n                            device: str | None = None) -> np.ndarray:\n    \"\"\"Top-k_max neighbor indices for each row, self excluded. [N, k_max]\"\"\"\n    return neighbors(embeddings, embeddings, k_max, exclude_self=True, device=device)\n\n\ndef recall_at_k(nbrs: np.ndarray, labels: np.ndarray, ks=(1, 5, 10)) -> dict:\n    hits = labels[nbrs] == labels[:, None]\n    return {f\"R@{k}\": float(hits[:, :k].any(axis=1).mean()) for k in ks}\n\n\ndef mean_average_precision(nbrs: np.ndarray, labels: np.ndarray, k: int = 100) -> float:\n    \"\"\"mAP@k over queries that have at least one relevant item.\"\"\"\n    rel = (labels[nbrs[:, :k]] == labels[:, None]).astype(np.float32)\n    # per-class counts (minus the query itself) for the normalizer\n    counts = np.bincount(labels)\n    n_rel = np.minimum(counts[labels] - 1, k)\n\n    cum_hits = np.cumsum(rel, axis=1)\n    ranks = np.arange(1, k + 1)\n    precision_at_hit = (cum_hits / ranks) * rel\n    valid = n_rel > 0\n    ap = precision_at_hit[valid].sum(axis=1) / n_rel[valid]\n    return float(ap.mean())\n\n\ndef metrics_from_neighbors(nbrs: np.ndarray, labels: np.ndarray,\n                           ks=(1, 5, 10)) -> dict:\n    m = recall_at_k(nbrs, labels, ks)\n    m[\"mAP@100\"] = mean_average_precision(nbrs, labels)\n    return m\n\n\ndef evaluate(embeddings: np.ndarray, labels: np.ndarray, ks=(1, 5, 10),\n             device: str | None = None) -> dict:\n    nbrs = leave_one_out_neighbors(embeddings, k_max=max(max(ks), 100), device=device)\n    return metrics_from_neighbors(nbrs, labels, ks)\n",
"vpse/retrieval/index.py": "\"\"\"Embed a dataset and build/query a FAISS index.\n\nEmbeddings are L2-normalized, so inner product == cosine similarity.\n\"\"\"\nfrom pathlib import Path\n\nimport faiss\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\n\n\n@torch.no_grad()\ndef embed_dataset(model, dataset, device: str, batch_size: int = 128,\n                  num_workers: int = 2) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Returns (embeddings [N, d] float32, labels [N]).\"\"\"\n    model.eval().to(device)\n    loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers)\n    embs, labels = [], []\n    for x, y in tqdm(loader, desc=\"embedding\"):\n        embs.append(model(x.to(device)).cpu().numpy())\n        labels.append(y.numpy())\n    return np.concatenate(embs).astype(\"float32\"), np.concatenate(labels)\n\n\ndef build_index(embeddings: np.ndarray) -> faiss.Index:\n    index = faiss.IndexFlatIP(embeddings.shape[1])\n    index.add(embeddings)\n    return index\n\n\ndef search(index: faiss.Index, queries: np.ndarray, k: int):\n    \"\"\"Returns (similarities [Nq, k], indices [Nq, k]).\"\"\"\n    return index.search(queries.astype(\"float32\"), k)\n\n\ndef save_embeddings(path: Path, embeddings: np.ndarray, labels: np.ndarray):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(path, embeddings=embeddings, labels=labels)\n\n\ndef load_embeddings(path: Path):\n    d = np.load(path)\n    return d[\"embeddings\"], d[\"labels\"]\n",
"vpse/retrieval/rerank.py": "\"\"\"Re-ranking by alpha-weighted query expansion.\n\nEach query is replaced by a similarity-weighted blend of itself and its top-k\nneighbours, then the search is repeated. Neighbours that the model is already\nconfident about pull the query toward the right region of the space; the alpha\nexponent suppresses weakly-matching ones. Cheap (one extra search) and needs no\ntraining, which is why it is the usual first re-ranking step to try.\n\nDefault k is deliberately small: SOP products carry roughly five images each, so\na query has at most ~4 true positives. Expanding over a larger neighbourhood\nmixes in other products and measurably hurts -- on a 1108-image probe, k=10\ndropped R@1 from 0.754 to 0.689 while k<=2 left it unchanged.\n\"\"\"\nimport numpy as np\n\nfrom vpse.retrieval.eval import metrics_from_neighbors, neighbors\n\n\ndef expand_queries(queries: np.ndarray, gallery: np.ndarray, nbrs: np.ndarray,\n                   k: int = 2, alpha: float = 3.0) -> np.ndarray:\n    \"\"\"alpha-QE: q' = normalize(q + sum_i sim_i^alpha * g_i) over top-k.\"\"\"\n    top = nbrs[:, :k]                                  # [Nq, k]\n    picked = gallery[top]                              # [Nq, k, d]\n    sims = np.einsum(\"qd,qkd->qk\", queries, picked)\n    weights = np.clip(sims, 0.0, None) ** alpha\n    expanded = queries + (weights[..., None] * picked).sum(axis=1)\n    norms = np.linalg.norm(expanded, axis=1, keepdims=True)\n    return (expanded / np.maximum(norms, 1e-12)).astype(\"float32\")\n\n\ndef rerank_leave_one_out(embeddings: np.ndarray, labels: np.ndarray,\n                         k: int = 2, alpha: float = 3.0, ks=(1, 5, 10),\n                         device: str | None = None):\n    \"\"\"Returns (metrics, neighbours) after one round of query expansion.\"\"\"\n    k_max = max(max(ks), 100)\n    base = neighbors(embeddings, embeddings, k_max, exclude_self=True, device=device)\n    expanded = expand_queries(embeddings, embeddings, base, k=k, alpha=alpha)\n    new = neighbors(expanded, embeddings, k_max, exclude_self=True, device=device)\n    return metrics_from_neighbors(new, labels, ks), new\n",
"vpse/retrieval/tta.py": "\"\"\"Test-time augmentation for embeddings.\n\nAveraging embeddings over several views of the same image smooths out crop and\norientation sensitivity. Views are produced on the batch tensor, so the dataset\nand its transform stay untouched.\n\"\"\"\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\n\nimport numpy as np\n\nVIEWS = (\"identity\", \"hflip\", \"zoom\")\n\n\ndef _apply_view(x: torch.Tensor, view: str) -> torch.Tensor:\n    if view == \"identity\":\n        return x\n    if view == \"hflip\":\n        return torch.flip(x, dims=[3])\n    if view == \"zoom\":\n        # centre 80% re-scaled back to full size\n        h, w = x.shape[-2:]\n        dh, dw = int(h * 0.1), int(w * 0.1)\n        return F.interpolate(x[:, :, dh:h - dh, dw:w - dw], size=(h, w),\n                             mode=\"bilinear\", align_corners=False)\n    raise ValueError(view)\n\n\n@torch.no_grad()\ndef embed_dataset_tta(model, dataset, device: str, views=VIEWS,\n                      batch_size: int = 128, num_workers: int = 2):\n    \"\"\"Returns (embeddings [N, d] float32, labels [N]), averaged over views.\"\"\"\n    model.eval().to(device)\n    loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers)\n    embs, labels = [], []\n    for x, y in tqdm(loader, desc=f\"embedding (TTA x{len(views)})\"):\n        x = x.to(device)\n        acc = None\n        for v in views:\n            e = model(_apply_view(x, v))\n            acc = e if acc is None else acc + e\n        embs.append(F.normalize(acc, dim=1).cpu().numpy())\n        labels.append(y.numpy())\n    return np.concatenate(embs).astype(\"float32\"), np.concatenate(labels)\n",
"vpse/train.py": "\"\"\"Phase 2 training loop. Runs locally (small batches) or on Kaggle GPU.\n\nUsage:\n    python -m vpse.train --loss triplet_hard --epochs 30\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom vpse.config import Config\nfrom vpse.data.samplers import PKSampler\nfrom vpse.data.sop import SOPDataset, eval_transform, train_transform\nfrom vpse.losses.arcface import ArcFaceHead\nfrom vpse.losses.triplet import triplet_batch_hard, triplet_random\nfrom vpse.models.embedder import Embedder\nfrom vpse.retrieval.eval import evaluate\nfrom vpse.retrieval.index import embed_dataset\n\n\ndef main(cfg: Config):\n    device = cfg.device if torch.cuda.is_available() else \"cpu\"\n    train_ds = SOPDataset(cfg.data_root, \"train\", train_transform(cfg.image_size))\n    test_ds = SOPDataset(cfg.data_root, \"test\", eval_transform(cfg.image_size))\n\n    sampler = PKSampler(train_ds.labels, cfg.batch_p, cfg.batch_k)\n    loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=cfg.num_workers,\n                        pin_memory=True)\n\n    model = Embedder(cfg.embedding_dim, freeze_backbone=cfg.freeze_backbone).to(device)\n    params = [{\"params\": model.head.parameters(), \"lr\": cfg.lr_head},\n              {\"params\": model.backbone.parameters(), \"lr\": cfg.lr_backbone}]\n\n    arcface = None\n    if cfg.loss == \"arcface\":\n        n_classes = int(train_ds.labels.max()) + 1\n        arcface = ArcFaceHead(cfg.embedding_dim, n_classes,\n                              cfg.arcface_scale, cfg.arcface_margin).to(device)\n        params.append({\"params\": arcface.parameters(), \"lr\": cfg.lr_head})\n\n    opt = torch.optim.AdamW(params, weight_decay=cfg.weight_decay)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)\n    scaler = torch.amp.GradScaler(enabled=device == \"cuda\")\n\n    best_r1 = 0.0\n    cfg.results_dir.mkdir(parents=True, exist_ok=True)\n    for epoch in range(cfg.epochs):\n        model.train()\n        running = 0.0\n        for step, (x, y) in enumerate(loader):\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            with torch.amp.autocast(device_type=\"cuda\", enabled=device == \"cuda\"):\n                emb = model(x)\n                if cfg.loss == \"triplet_random\":\n                    loss = triplet_random(emb, y, cfg.triplet_margin)\n                elif cfg.loss == \"triplet_hard\":\n                    loss = triplet_batch_hard(emb, y, cfg.triplet_margin)\n                elif cfg.loss == \"arcface\":\n                    loss = arcface(emb, y)\n                else:\n                    raise ValueError(cfg.loss)\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            running += loss.item()\n            if step % 50 == 0:\n                print(f\"epoch {epoch} step {step}/{len(loader)} \"\n                      f\"loss {running / (step + 1):.4f}\")\n        sched.step()\n\n        is_last = epoch == cfg.epochs - 1\n        if not (is_last or (epoch + 1) % cfg.eval_every == 0):\n            continue\n\n        embs, labels = embed_dataset(model, test_ds, device,\n                                     num_workers=cfg.num_workers)\n        metrics = evaluate(embs, labels, cfg.recall_ks, device=device)\n        print(f\"epoch {epoch}: {metrics}\")\n        if metrics[\"R@1\"] > best_r1:\n            best_r1 = metrics[\"R@1\"]\n            torch.save(model.state_dict(), cfg.results_dir / f\"best_{cfg.loss}.pt\")\n            (cfg.results_dir / f\"best_{cfg.loss}.json\").write_text(\n                json.dumps({\"epoch\": epoch, **metrics}, indent=2))\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--loss\", default=\"triplet_hard\",\n                    choices=[\"triplet_random\", \"triplet_hard\", \"arcface\"])\n    ap.add_argument(\"--epochs\", type=int, default=30)\n    ap.add_argument(\"--data-root\", default=\"data/Stanford_Online_Products\")\n    ap.add_argument(\"--batch-p\", type=int, default=16)\n    ap.add_argument(\"--batch-k\", type=int, default=4)\n    args = ap.parse_args()\n    main(Config(loss=args.loss, epochs=args.epochs, data_root=Path(args.data_root),\n                batch_p=args.batch_p, batch_k=args.batch_k))\n"
}
''')
_SRC = pathlib.Path('/kaggle/working/src')
for _rel, _text in _FILES.items():
    _p = _SRC / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(_text, encoding='utf-8')
sys.path.insert(0, str(_SRC))
print(f'wrote {len(_FILES)} modules to {_SRC}')

In [ ]:
import glob
from pathlib import Path
import torch

hits = glob.glob('/kaggle/input/**/Ebay_train.txt', recursive=True)
assert hits, 'attach the Stanford Online Products dataset'
DATA_ROOT = Path(hits[0]).parent

ckpts = sorted(glob.glob('/kaggle/input/**/best_*.pt', recursive=True))
if not ckpts:
    print('No checkpoint found under /kaggle/input. Attached inputs:')
    for r in sorted(Path('/kaggle/input').glob('*')):
        print(' ', r.name)
    raise SystemExit('Attach the Phase 2 run output (Add Input -> Your Work) so '
                     'best_triplet_hard.pt is visible.')
CKPT = next((c for c in ckpts if 'triplet_hard' in c), ckpts[0])  # Phase 2 winner

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('data  :', DATA_ROOT)
print('ckpt  :', CKPT)
print('device:', DEVICE)

## The trained model

Re-embed the test split with the Phase 2 winner. R@1 should reproduce the
number recorded in the README; if it does not, the wrong checkpoint is
attached.

In [ ]:
import json
from vpse.data.sop import SOPDataset, eval_transform
from vpse.models.embedder import Embedder
from vpse.retrieval.index import embed_dataset
from vpse.retrieval.eval import leave_one_out_neighbors, metrics_from_neighbors

test_ds = SOPDataset(DATA_ROOT, 'test', eval_transform())

model = Embedder(embedding_dim=512)          # must match the training config
state = torch.load(CKPT, map_location='cpu')
missing, unexpected = model.load_state_dict(state, strict=False)
assert not missing and not unexpected, (missing, unexpected)
print('checkpoint loaded cleanly')

embs, labels = embed_dataset(model, test_ds, DEVICE, batch_size=256, num_workers=4)
base_nbrs = leave_one_out_neighbors(embs, k_max=100, device=DEVICE)
base = metrics_from_neighbors(base_nbrs, labels)
print('trained model:', json.dumps(base, indent=2))
print('sanity check: R@1 should land near the Phase 2 value of 0.7277')

## Where do the errors come from?

SOP labels each image with a product *and* a category. When top-1 is the wrong
product but the **same category**, the model found the right kind of thing and
picked the wrong instance — the look-alike failure. That share is what Phase 3
attacks.

In [ ]:
from vpse.analysis.errors import error_breakdown, hard_queries, recall_on_subset
from vpse.analysis.grids import retrieval_grid
from IPython.display import Image as IPyImage

breakdown = error_breakdown(base_nbrs, labels, test_ds.super_labels)
print(json.dumps(breakdown, indent=2))

hard = hard_queries(base_nbrs, labels, test_ds.super_labels)
print(f'look-alike failures: {len(hard):,} queries')
print('their recall:', recall_on_subset(base_nbrs, labels, hard))

grid = retrieval_grid(test_ds, base_nbrs, labels, hard[:6],
                      Path('/kaggle/working/lookalike_failures.png'),
                      title='Look-alike failures: top-1 is the wrong product')
IPyImage(str(grid))

## Fix attempt 1 — test-time augmentation

Average each embedding over several views. On a small untrained probe this was
roughly neutral overall while recovering some look-alike failures, so the
question is whether it holds up on the trained model.

In [ ]:
from vpse.retrieval.tta import embed_dataset_tta

tta_results = {}
for views in (('identity', 'hflip'), ('identity', 'hflip', 'zoom')):
    e, _ = embed_dataset_tta(model, test_ds, DEVICE, views=views,
                             batch_size=256, num_workers=4)
    nb = leave_one_out_neighbors(e, k_max=100, device=DEVICE)
    m = metrics_from_neighbors(nb, labels)
    m['hard_R@1'] = recall_on_subset(nb, labels, hard)['R@1']
    tta_results['TTA ' + '+'.join(views)] = (m, nb)
    print('+'.join(views), {k: round(v, 4) for k, v in m.items()})

## Fix attempt 2 — re-ranking by query expansion

Blend each query with its top-k neighbours and search again. `k` matters: SOP
products have only ~5 images, so a large neighbourhood pulls in other products.
On the untrained probe, k=10 cost 6.5 points of R@1 — hence the sweep.

In [ ]:
from vpse.retrieval.eval import neighbors as nn_search
from vpse.retrieval.rerank import expand_queries

header = f"{'k':>3} {'alpha':>6} {'R@1':>8} {'R@5':>8} {'mAP':>8} {'hardR@1':>9}"
print(header)
qe_results = {}
for k in (1, 2, 3, 5):
    for a in (1.0, 3.0):
        exp = expand_queries(embs, embs, base_nbrs, k=k, alpha=a)
        nb = nn_search(exp, embs, 100, exclude_self=True, device=DEVICE)
        m = metrics_from_neighbors(nb, labels)
        m['hard_R@1'] = recall_on_subset(nb, labels, hard)['R@1']
        qe_results[f'alpha-QE k={k} a={a}'] = (m, nb)
        print(f"{k:>3} {a:>6.1f} {m['R@1']:>8.4f} {m['R@5']:>8.4f} "
              f"{m['mAP@100']:>8.4f} {m['hard_R@1']:>9.4f}")

## Before / after

Keep whatever actually won. If nothing beats the plain trained model, that is
the finding — report it rather than burying it.

In [ ]:
rows = {'trained (plain)': ({**base, 'hard_R@1': 0.0}, base_nbrs)}
rows.update(tta_results)
rows.update(qe_results)

best = max(rows, key=lambda n: rows[n][0]['R@1'])
print(f"{'variant':<28} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'mAP':>8} {'hardR@1':>9}")
summary = {}
for name, (m, nb) in rows.items():
    summary[name] = {k: float(v) for k, v in m.items()}
    star = ' *' if name == best else ''
    print(f"{name:<28} {m['R@1']:>8.4f} {m['R@5']:>8.4f} {m['R@10']:>8.4f} "
          f"{m['mAP@100']:>8.4f} {m['hard_R@1']:>9.4f}{star}")

print(f'best by R@1: {best}')
Path('/kaggle/working/phase3.json').write_text(json.dumps(
    {'breakdown': breakdown, 'n_hard': int(len(hard)),
     'variants': summary, 'best': best}, indent=2))
print('wrote phase3.json')